# Capítulo 13 — Importação e exportação

Execute em ordem e salve sua cópia. As definições são independentes dos arquivos do computador. Esta etapa não contém recorrência.


In [ ]:
"""Capítulo 8: prioridades, filtros e prazos, com dados em memória."""

from copy import deepcopy
from datetime import date


def cadastrar_tarefa(tarefas, titulo, prioridade="media", prazo=None):
    if not isinstance(titulo, str) or not titulo.strip():
        return "Informe um título com pelo menos um caractere que não seja espaço."
    if prioridade not in ("baixa", "media", "alta"):
        return "Prioridade inválida. Use baixa, media ou alta."
    try:
        data_prazo = interpretar_prazo(prazo)
    except ValueError as erro:
        return str(erro)
    tarefas.append({
        "titulo": titulo,
        "concluida": False,
        "prioridade": prioridade,
        "prazo": data_prazo,
    })
    return "Tarefa cadastrada."


def concluir_tarefa(tarefas, indice):
    if indice < 0 or indice >= len(tarefas):
        return "Tarefa não encontrada."
    tarefas[indice]["concluida"] = True
    return "Tarefa concluída."




def filtrar_tarefas(tarefas, prioridade=None, concluida=None):
    if prioridade is not None and prioridade not in ("baixa", "media", "alta"):
        raise ValueError("Prioridade de filtro inválida.")
    if concluida is not None and type(concluida) is not bool:
        raise ValueError("Conclusão de filtro deve ser booleana.")
    resultado = []
    for tarefa in tarefas:
        if prioridade is not None and tarefa.get("prioridade", "media") != prioridade:
            continue
        if concluida is not None and tarefa["concluida"] != concluida:
            continue
        resultado.append(deepcopy(tarefa))
    return resultado


def interpretar_prazo(texto):
    if texto is None:
        return None
    if not isinstance(texto, str):
        raise ValueError("Prazo deve ser texto no formato AAAA-MM-DD.")
    try:
        prazo = date.fromisoformat(texto)
    except ValueError:
        raise ValueError("Prazo inválido. Use AAAA-MM-DD.") from None
    if prazo.isoformat() != texto:
        raise ValueError("Prazo inválido. Use AAAA-MM-DD.")
    return prazo


def esta_atrasada(tarefa, hoje):
    prazo = tarefa.get("prazo")
    return (
        not tarefa["concluida"]
        and prazo is not None
        and prazo < hoje
    )

"""Apresentação textual preservada desde o primeiro exemplo."""


def listar_tarefas(tarefas):
    if not tarefas:
        return "Nenhuma tarefa cadastrada."
    linhas = []
    for indice, tarefa in enumerate(tarefas):
        marca = "x" if tarefa["concluida"] else " "
        linhas.append(f"{indice}: [{marca}] {tarefa['titulo']}")
    return "\n".join(linhas)

"""Importação/exportação de texto JSON; não grava arquivos por conta própria."""

import json


LIMITE_BYTES = 256000
LIMITE_REGISTROS = 1000
CAMPOS = {"titulo", "concluida", "prioridade", "prazo"}


def objeto_sem_chaves_duplicadas(pares):
    objeto = {}
    for chave, valor in pares:
        if chave in objeto:
            raise ValueError(f"Chave duplicada: {chave}.")
        objeto[chave] = valor
    return objeto


def validar_registro(registro):
    if not isinstance(registro, dict):
        raise ValueError("Cada tarefa deve ser um objeto.")
    if set(registro) - CAMPOS:
        raise ValueError("Campos desconhecidos.")
    if "titulo" not in registro or "concluida" not in registro:
        raise ValueError("Título e conclusão são obrigatórios.")
    if type(registro["concluida"]) is not bool:
        raise ValueError("Conclusão deve ser booleana.")
    temporarias = []
    mensagem = cadastrar_tarefa(
        temporarias,
        registro["titulo"],
        prioridade=registro.get("prioridade", "media"),
        prazo=registro.get("prazo"),
    )
    if mensagem != "Tarefa cadastrada.":
        raise ValueError(mensagem)
    tarefa = temporarias[0]
    tarefa["concluida"] = registro["concluida"]
    return tarefa


def importar_json(tarefas, texto):
    if not isinstance(texto, str):
        return "Importação recusada: forneça texto JSON."
    try:
        tamanho = len(texto.encode("utf-8"))
    except UnicodeError:
        return "Importação recusada: texto Unicode inválido."
    if tamanho > LIMITE_BYTES:
        return "Importação recusada: limite de tamanho excedido."
    try:
        registros = json.loads(texto, object_pairs_hook=objeto_sem_chaves_duplicadas)
    except (ValueError, RecursionError):
        return "Importação recusada: JSON inválido ou chave duplicada."
    if not isinstance(registros, list) or len(registros) > LIMITE_REGISTROS:
        return "Importação recusada: use uma lista com até 1000 registros."
    temporarias = []
    for indice, registro in enumerate(registros):
        try:
            temporarias.append(validar_registro(registro))
        except ValueError as erro:
            return f"Importação recusada no registro {indice}: {erro}"
    tarefas.extend(temporarias)
    return f"Importação concluída: {len(temporarias)} tarefa(s)."


def exportar_json(tarefas):
    registros = []
    for tarefa in tarefas:
        registro = dict(tarefa)
        prazo = registro.get("prazo")
        if prazo is not None:
            registro["prazo"] = prazo.isoformat()
        registros.append(registro)
    return json.dumps(registros, ensure_ascii=False, indent=2, allow_nan=False)


## Entrada completa e resultado

A importação espera uma lista JSON e deve criar um registro.


In [ ]:
tarefas = []
texto = '[{"titulo":"Revisar manuscrito","concluida":false,"prioridade":"alta","prazo":"2026-10-20"}]'
print(importar_json(tarefas, texto))
print(len(tarefas))
print(exportar_json(tarefas))


## Conferir preservação dos dados


In [ ]:
antes = deepcopy(tarefas)
registros = [{"titulo": str(i), "concluida": False} for i in range(3)]
registros[2]["prioridade"] = "urgente"
assert "registro 2" in importar_json(tarefas, json.dumps(registros))
assert tarefas == antes
registros[2]["prioridade"] = "alta"
assert importar_json(tarefas, json.dumps(registros)).startswith("Importação concluída")
assert len(tarefas) == 4
for texto_invalido in ('{}', '[{"titulo":"A","concluida":"sim"}]', '[{"titulo":"A","concluida":false,"extra":1}]'):
    antes = deepcopy(tarefas)
    assert importar_json(tarefas, texto_invalido).startswith("Importação recusada")
    assert tarefas == antes
print("Importacao integral e recusas conferidas, sem recorrencia.")
